In [1]:
import sys
sys.path.append('../../')

In [2]:
from DB import IO_mac
import pandas as pd
import numpy as np
db = IO_mac.Execute()

In [3]:
country = 'US'
sector = 'Information Technology'
symbol = '069500' if country=='KR' else 'SPY'
def set_period(start_term, end_term):
    term = pd.period_range(start=start_term, end=end_term, freq='3M')
    release_date = (term.end_time + pd.DateOffset(months=2)).date
    return term, release_date

In [4]:
start_term = '2006-01-01'
end_term = '2020-09-30'
term, release_date = set_period(start_term, end_term)
print(term)

PeriodIndex(['2006-01', '2006-04', '2006-07', '2006-10', '2007-01', '2007-04',
             '2007-07', '2007-10', '2008-01', '2008-04', '2008-07', '2008-10',
             '2009-01', '2009-04', '2009-07', '2009-10', '2010-01', '2010-04',
             '2010-07', '2010-10', '2011-01', '2011-04', '2011-07', '2011-10',
             '2012-01', '2012-04', '2012-07', '2012-10', '2013-01', '2013-04',
             '2013-07', '2013-10', '2014-01', '2014-04', '2014-07', '2014-10',
             '2015-01', '2015-04', '2015-07', '2015-10', '2016-01', '2016-04',
             '2016-07', '2016-10', '2017-01', '2017-04', '2017-07', '2017-10',
             '2018-01', '2018-04', '2018-07', '2018-10', '2019-01', '2019-04',
             '2019-07', '2019-10', '2020-01', '2020-04', '2020-07'],
            dtype='period[3M]', freq='3M')


In [5]:
release_date

array([datetime.date(2006, 5, 31), datetime.date(2006, 8, 30),
       datetime.date(2006, 11, 30), datetime.date(2007, 2, 28),
       datetime.date(2007, 5, 31), datetime.date(2007, 8, 30),
       datetime.date(2007, 11, 30), datetime.date(2008, 2, 29),
       datetime.date(2008, 5, 31), datetime.date(2008, 8, 30),
       datetime.date(2008, 11, 30), datetime.date(2009, 2, 28),
       datetime.date(2009, 5, 31), datetime.date(2009, 8, 30),
       datetime.date(2009, 11, 30), datetime.date(2010, 2, 28),
       datetime.date(2010, 5, 31), datetime.date(2010, 8, 30),
       datetime.date(2010, 11, 30), datetime.date(2011, 2, 28),
       datetime.date(2011, 5, 31), datetime.date(2011, 8, 30),
       datetime.date(2011, 11, 30), datetime.date(2012, 2, 29),
       datetime.date(2012, 5, 31), datetime.date(2012, 8, 30),
       datetime.date(2012, 11, 30), datetime.date(2013, 2, 28),
       datetime.date(2013, 5, 31), datetime.date(2013, 8, 30),
       datetime.date(2013, 11, 30), datetime.dat

In [6]:
acct_is_ttm = {
    'revenue': ['Revenue', 'ISQ'],
    'operating_income': ['Operating Income', 'ISQ'],
    'net_income': ['Net Income', 'ISQ'],
    'EPS': ['EPS - Earnings Per Share', 'ISQ'],
}

In [7]:
def fn_quarterly(acct, country='US', sector=None):
    query = 'SELECT M.issue_code, '
    for k, v in acct.items():
        for t in term:
            query += '{}_{}, '.format(k, t.strftime('%y%qQ'))
    query = query[:-2]
    query += ' FROM issue_master M, '
    for k, v in acct.items():
        query += '(SELECT issue_code, '
        for t in term:
            query += ' MAX(CASE WHEN term BETWEEN "{}" AND "{}" THEN vlu END) AS {}_{}, '.format(t.start_time.date(), t.end_time.date(), k, t.strftime('%y%qQ'))
        query = query[:-2]    
        query += ' FROM _fn_stmt WHERE title="{}" AND category="{}" GROUP BY issue_code) {}, '.format(v[0], v[1], k)
    query = query[:-2]  
    query += ' WHERE M.country="{}" '.format(country)
    for k, v in acct.items():
        query += ' AND M.issue_code = {}.issue_code '.format(k)
    query += 'AND M.sector="{}" '.format(sector) if sector else ''
    #print(query)
    df = db.query_df(query)
    df.set_index('issue_code', inplace=True)
    return df

In [8]:
qtr1 = fn_quarterly(acct_is_ttm, country, sector)
qtr1

,revenue_061Q,revenue_062Q,revenue_063Q,revenue_064Q,revenue_071Q,revenue_072Q,revenue_073Q,revenue_074Q,revenue_081Q,revenue_082Q,...,EPS_182Q,EPS_183Q,EPS_184Q,EPS_191Q,EPS_192Q,EPS_193Q,EPS_194Q,EPS_201Q,EPS_202Q,EPS_203Q
issue_code,,,,,,,,,,,,,,,,,,,,,
AAPL,4359.000000,4370.000000,4837.000000,7115.000000,5264.000000,5410.000000,6789.000000,9608.000000,7512.000000,7464.000000,...,0.585,0.7375,1.045,0.615,0.545,0.7675,1.2475,0.6375,0.645,NaN
ACIW,89.833000,84.764000,NaN,93.289001,89.947998,98.109001,84.872002,101.281998,90.663002,109.219002,...,-0.130,0.1300,0.760,-0.220,0.050,0.2700,0.4700,-0.2100,0.120,NaN
ACLS,97.920998,117.639999,122.817001,123.338997,97.526001,110.072998,107.553001,89.648003,84.893997,76.889000,...,0.430,0.2600,0.250,0.180,0.020,0.0200,0.2800,0.3300,0.390,NaN
ACN,4491.111816,4805.327148,4388.910156,5166.358887,5169.353027,5543.684082,5573.350098,6101.957031,6057.623047,6593.201172,...,1.600,1.5800,1.960,1.730,1.930,1.7400,2.0900,1.9100,1.900,1.99
ADBE,655.478027,635.455994,602.190979,682.174988,649.406982,745.577026,851.685974,911.211121,890.445007,886.885986,...,1.330,1.3400,1.360,1.360,1.290,1.6100,1.7400,1.9600,2.270,1.97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WDC,1129.000000,1085.000000,1264.000000,1428.000000,1410.000000,1366.000000,1766.000000,2204.000000,2111.000000,1993.000000,...,2.550,1.7100,-1.680,-1.990,-0.620,-0.9300,-0.4700,0.0600,0.500,NaN
WU,1043.000000,1113.599976,1140.400024,1173.199951,1131.000000,1202.900024,1257.199951,1309.099976,1265.900024,1347.099976,...,0.470,0.4600,0.480,0.390,1.420,0.3200,0.3300,0.4200,0.390,NaN
XLNX,472.337097,481.362000,467.179993,450.725006,443.471985,445.911987,444.894012,474.806000,475.759888,488.246002,...,0.740,0.8400,0.930,0.960,0.940,0.8900,0.6400,0.6400,0.380,0.79


In [9]:
qtr1.fillna(value=np.nan, inplace=True)

In [10]:
ttm1 = qtr1.rolling(4, axis=1).sum()
ttm1

,revenue_061Q,revenue_062Q,revenue_063Q,revenue_064Q,revenue_071Q,revenue_072Q,revenue_073Q,revenue_074Q,revenue_081Q,revenue_082Q,...,EPS_182Q,EPS_183Q,EPS_184Q,EPS_191Q,EPS_192Q,EPS_193Q,EPS_194Q,EPS_201Q,EPS_202Q,EPS_203Q
issue_code,,,,,,,,,,,,,,,,,,,,,
AAPL,NaN,NaN,NaN,20681.000000,21586.000000,22626.000000,24578.000000,27071.000000,29319.000000,31373.000000,...,2.760000e+00,2.9775,3.05,2.9825,2.9425,2.9725,3.175,3.1975,3.2975,NaN
ACIW,NaN,NaN,NaN,NaN,NaN,NaN,366.218002,374.210999,374.926003,386.036003,...,1.303852e-08,0.1000,0.59,0.5400,0.7200,0.8600,0.570,0.5800,0.6500,NaN
ACLS,NaN,NaN,NaN,461.716995,461.321999,453.754997,438.490997,404.800003,392.167999,358.984001,...,3.930000e+00,3.8400,1.35,1.1200,0.7100,0.4700,0.500,0.6500,1.0200,NaN
ACN,NaN,NaN,NaN,18851.708008,19529.949219,20268.306152,21452.746094,22388.344238,23276.614258,24326.131348,...,6.240000e+00,6.3400,6.51,6.8700,7.2000,7.3600,7.490,7.6700,7.6400,7.89
ADBE,NaN,NaN,NaN,2575.299988,2569.228943,2679.349976,2928.844971,3157.881104,3398.919128,3540.228088,...,4.330000e+00,4.8300,5.20,5.3900,5.3500,5.6200,6.000,6.6000,7.5800,7.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WDC,NaN,NaN,NaN,4906.000000,5187.000000,5468.000000,5970.000000,6746.000000,7447.000000,8074.000000,...,2.200000e+00,1.6800,2.78,0.5900,-2.5800,-5.2200,-4.010,-1.9600,-0.8400,NaN
WU,NaN,NaN,NaN,4470.199951,4558.199951,4647.500000,4764.299927,4900.199951,5035.099976,5179.299927,...,-9.400001e-01,-0.9900,1.87,1.8000,2.7500,2.6100,2.460,2.4900,1.4600,NaN
XLNX,NaN,NaN,NaN,1871.604095,1842.738983,1807.288971,1785.002991,1809.083984,1841.371887,1883.705902,...,1.950000e+00,2.1200,3.10,3.4700,3.6700,3.7200,3.430,3.1100,2.5500,2.45


In [11]:
df = qtr1.merge(ttm1, on='issue_code', suffixes=('','_ttm'))
df

,revenue_061Q,revenue_062Q,revenue_063Q,revenue_064Q,revenue_071Q,revenue_072Q,revenue_073Q,revenue_074Q,revenue_081Q,revenue_082Q,...,EPS_182Q_ttm,EPS_183Q_ttm,EPS_184Q_ttm,EPS_191Q_ttm,EPS_192Q_ttm,EPS_193Q_ttm,EPS_194Q_ttm,EPS_201Q_ttm,EPS_202Q_ttm,EPS_203Q_ttm
issue_code,,,,,,,,,,,,,,,,,,,,,
AAPL,4359.000000,4370.000000,4837.000000,7115.000000,5264.000000,5410.000000,6789.000000,9608.000000,7512.000000,7464.000000,...,2.760000e+00,2.9775,3.05,2.9825,2.9425,2.9725,3.175,3.1975,3.2975,NaN
ACIW,89.833000,84.764000,NaN,93.289001,89.947998,98.109001,84.872002,101.281998,90.663002,109.219002,...,1.303852e-08,0.1000,0.59,0.5400,0.7200,0.8600,0.570,0.5800,0.6500,NaN
ACLS,97.920998,117.639999,122.817001,123.338997,97.526001,110.072998,107.553001,89.648003,84.893997,76.889000,...,3.930000e+00,3.8400,1.35,1.1200,0.7100,0.4700,0.500,0.6500,1.0200,NaN
ACN,4491.111816,4805.327148,4388.910156,5166.358887,5169.353027,5543.684082,5573.350098,6101.957031,6057.623047,6593.201172,...,6.240000e+00,6.3400,6.51,6.8700,7.2000,7.3600,7.490,7.6700,7.6400,7.89
ADBE,655.478027,635.455994,602.190979,682.174988,649.406982,745.577026,851.685974,911.211121,890.445007,886.885986,...,4.330000e+00,4.8300,5.20,5.3900,5.3500,5.6200,6.000,6.6000,7.5800,7.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WDC,1129.000000,1085.000000,1264.000000,1428.000000,1410.000000,1366.000000,1766.000000,2204.000000,2111.000000,1993.000000,...,2.200000e+00,1.6800,2.78,0.5900,-2.5800,-5.2200,-4.010,-1.9600,-0.8400,NaN
WU,1043.000000,1113.599976,1140.400024,1173.199951,1131.000000,1202.900024,1257.199951,1309.099976,1265.900024,1347.099976,...,-9.400001e-01,-0.9900,1.87,1.8000,2.7500,2.6100,2.460,2.4900,1.4600,NaN
XLNX,472.337097,481.362000,467.179993,450.725006,443.471985,445.911987,444.894012,474.806000,475.759888,488.246002,...,1.950000e+00,2.1200,3.10,3.4700,3.6700,3.7200,3.430,3.1100,2.5500,2.45


In [12]:
acct_ratio = {
    'current_ratio': ['Current Ratio', 'RQ'],
    'debt_to_equity': ['Debt/Equity Ratio', 'RQ'],
    'ROE': ['ROE - Return On Equity', 'RQ'],
    'ROA': ['ROA - Return On Assets', 'RQ'],
    'BPS': ['Book Value Per Share', 'RQ'],
}

In [13]:
qtr2 = fn_quarterly(acct_ratio, country, sector)
qtr2

,current_ratio_061Q,current_ratio_062Q,current_ratio_063Q,current_ratio_064Q,current_ratio_071Q,current_ratio_072Q,current_ratio_073Q,current_ratio_074Q,current_ratio_081Q,current_ratio_082Q,...,BPS_182Q,BPS_183Q,BPS_184Q,BPS_191Q,BPS_192Q,BPS_193Q,BPS_194Q,BPS_201Q,BPS_202Q,BPS_203Q
issue_code,,,,,,,,,,,,,,,,,,,,,
AAPL,2.5328,2.4868,2.2519,2.2712,2.9223,2.6809,2.3659,2.4859,2.7752,3.0373,...,5.9339,5.633400,6.231300,5.744200,5.321500,5.091300,5.104400,4.534300,4.218200,NaN
ACIW,2.2375,2.0460,NaN,1.5343,1.4471,1.3152,1.1123,1.2078,1.1975,1.2447,...,8.1261,8.302500,9.026900,8.861100,9.020700,9.163100,9.742300,9.214500,9.373300,NaN
ACLS,2.3614,2.4858,2.4188,2.4984,4.2176,4.0519,4.5040,4.0284,2.3209,2.3110,...,11.9271,12.256100,12.541800,12.686800,12.554300,12.589000,12.871800,13.008900,13.310900,NaN
ACN,1.1734,1.2101,1.2457,1.2163,1.2439,1.2058,1.1587,1.2608,1.2595,1.3158,...,15.8253,16.765301,20.462400,21.499100,22.173401,23.299900,24.546000,24.961300,25.895201,NaN
ADBE,4.5432,4.0546,4.4765,4.2585,4.3583,3.7897,3.1015,3.0183,3.3250,3.5398,...,17.7507,18.122299,19.197901,20.207600,20.424101,21.123301,21.801201,21.666700,22.668800,24.4021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WDAY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.7084,8.180800,8.408900,8.821800,9.047600,9.694200,10.083400,10.717900,11.072400,11.9767
WDC,1.6908,1.7259,1.7281,1.7670,1.8167,1.7956,0.9935,1.1280,1.7597,1.7462,...,38.9561,39.359901,37.498299,34.860100,33.786400,32.100700,31.367901,30.809999,31.625799,NaN
WU,NaN,NaN,1.5570,1.4728,1.5419,1.5273,1.4890,1.5969,1.5553,1.5357,...,-1.0058,-0.936000,-0.702200,-0.864400,0.070900,-0.046900,-0.094500,-0.364300,-0.178600,NaN


In [14]:
df = df.merge(qtr2, on='issue_code', suffixes=('',''))
df.dropna(how='all', axis=1, inplace=True)
df

,revenue_061Q,revenue_062Q,revenue_063Q,revenue_064Q,revenue_071Q,revenue_072Q,revenue_073Q,revenue_074Q,revenue_081Q,revenue_082Q,...,BPS_182Q,BPS_183Q,BPS_184Q,BPS_191Q,BPS_192Q,BPS_193Q,BPS_194Q,BPS_201Q,BPS_202Q,BPS_203Q
issue_code,,,,,,,,,,,,,,,,,,,,,
AAPL,4359.000000,4370.000000,4837.000000,7115.000000,5264.000000,5410.000000,6789.000000,9608.000000,7512.000000,7464.000000,...,5.9339,5.633400,6.231300,5.744200,5.321500,5.091300,5.104400,4.534300,4.218200,NaN
ACIW,89.833000,84.764000,NaN,93.289001,89.947998,98.109001,84.872002,101.281998,90.663002,109.219002,...,8.1261,8.302500,9.026900,8.861100,9.020700,9.163100,9.742300,9.214500,9.373300,NaN
ACLS,97.920998,117.639999,122.817001,123.338997,97.526001,110.072998,107.553001,89.648003,84.893997,76.889000,...,11.9271,12.256100,12.541800,12.686800,12.554300,12.589000,12.871800,13.008900,13.310900,NaN
ACN,4491.111816,4805.327148,4388.910156,5166.358887,5169.353027,5543.684082,5573.350098,6101.957031,6057.623047,6593.201172,...,15.8253,16.765301,20.462400,21.499100,22.173401,23.299900,24.546000,24.961300,25.895201,NaN
ADBE,655.478027,635.455994,602.190979,682.174988,649.406982,745.577026,851.685974,911.211121,890.445007,886.885986,...,17.7507,18.122299,19.197901,20.207600,20.424101,21.123301,21.801201,21.666700,22.668800,24.4021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WDAY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.7084,8.180800,8.408900,8.821800,9.047600,9.694200,10.083400,10.717900,11.072400,11.9767
WDC,1129.000000,1085.000000,1264.000000,1428.000000,1410.000000,1366.000000,1766.000000,2204.000000,2111.000000,1993.000000,...,38.9561,39.359901,37.498299,34.860100,33.786400,32.100700,31.367901,30.809999,31.625799,NaN
WU,1043.000000,1113.599976,1140.400024,1173.199951,1131.000000,1202.900024,1257.199951,1309.099976,1265.900024,1347.099976,...,-1.0058,-0.936000,-0.702200,-0.864400,0.070900,-0.046900,-0.094500,-0.364300,-0.178600,NaN


In [15]:
df[(df['EPS_202Q']>df['EPS_192Q'])&(df['EPS_202Q_ttm']>df['EPS_192Q_ttm'])]

,revenue_061Q,revenue_062Q,revenue_063Q,revenue_064Q,revenue_071Q,revenue_072Q,revenue_073Q,revenue_074Q,revenue_081Q,revenue_082Q,...,BPS_182Q,BPS_183Q,BPS_184Q,BPS_191Q,BPS_192Q,BPS_193Q,BPS_194Q,BPS_201Q,BPS_202Q,BPS_203Q
issue_code,,,,,,,,,,,,,,,,,,,,,
AAPL,4359.000000,4370.000000,4837.000000,7115.000000,5264.000000,5410.000000,6789.000000,9608.000000,7512.000000,7464.000000,...,5.933900,5.633400,6.231300,5.7442,5.321500,5.091300,5.104400,4.534300,4.218200,NaN
ACLS,97.920998,117.639999,122.817001,123.338997,97.526001,110.072998,107.553001,89.648003,84.893997,76.889000,...,11.927100,12.256100,12.541800,12.6868,12.554300,12.589000,12.871800,13.008900,13.310900,NaN
ADBE,655.478027,635.455994,602.190979,682.174988,649.406982,745.577026,851.685974,911.211121,890.445007,886.885986,...,17.750700,18.122299,19.197901,20.2076,20.424101,21.123301,21.801201,21.666700,22.668800,24.402100
ADSK,423.799988,436.000000,449.600006,456.799988,497.399994,508.600006,525.900024,538.400024,598.999878,598.799988,...,-0.586800,-1.105100,-1.544700,-0.9613,-1.116900,-0.884800,-0.779400,-0.634000,-0.634600,0.306900
AKAM,90.824997,100.649002,111.495003,125.703003,139.274002,152.654007,161.240005,183.238007,187.018997,194.003998,...,20.348801,19.254299,19.593399,20.0805,20.781000,21.660299,22.579901,22.643299,24.007200,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.262700,3.714600,1.775300,2.6652,1.429000,-3.636100,-4.502400,-5.581200,-4.639100,NaN
VIAV,314.899994,318.200012,318.100006,366.299988,361.700012,350.700104,356.700012,399.200012,383.899994,372.200012,...,3.237400,3.112300,3.114500,3.1803,3.169400,3.078700,3.323000,2.968900,3.120200,NaN
VMW,NaN,156.440002,188.806000,NaN,258.695007,296.825012,357.816010,NaN,438.174988,456.127991,...,23.767700,25.475700,26.612200,7.0389,1.367400,12.696200,14.190000,16.779800,17.715700,18.805201


In [16]:
query = 'SELECT issue_code FROM issue_master '
query += 'WHERE active = "Y" '
query += 'AND country="{}" '.format(country)
query += 'AND sector="{}" '.format(sector)
query

'SELECT issue_code FROM issue_master WHERE active = "Y" AND country="US" AND sector="Information Technology" '

In [17]:
symbol = db.query_df(query).issue_code.to_list()

In [18]:
def price_close_daily_portfolio(cd, start_date, end_date):
    sql = 'SELECT trade_date '
    for c in cd:
        sql += ', max(case when issue_code="{c}" then price_close end) as "{c}" '.format(c=c)
    sql += 'FROM _price_daily_archive '
    sql += 'WHERE  issue_code in ( '
    for c in cd:
        sql += '"{}" '.format(c)
        if c is not cd[-1]:
            sql += ', '
    sql += ') '
    sql += 'AND trade_date >= "{}" '.format(start_date)
    sql += 'AND trade_date <= "{}" '.format(end_date)
    sql += 'GROUP BY trade_date '
    sql += 'ORDER BY trade_date ; '
    rows = db.query_df(sql)
    rows.set_index('trade_date', inplace=True)
    return rows

In [19]:
price_df = price_close_daily_portfolio(symbol, start_term, end_term)
price_df

,AAPL,ACIW,ACLS,ACN,ADBE,ADI,ADP,ADS,ADSK,ADTN,...,V,VIAV,VMW,VRSN,WDAY,WDC,WU,XLNX,XRX,ZM
trade_date,,,,,,,,,,,,,,,,,,,,,
2006-01-03,2.670000,NaN,NaN,29.299999,38.520000,36.700001,36.509998,36.959999,42.750000,NaN,...,NaN,NaN,NaN,21.559999,NaN,19.020000,NaN,25.770000,14.910000,NaN
2006-01-04,2.680000,NaN,NaN,29.459999,38.419998,37.040001,36.630001,37.849998,42.520000,NaN,...,NaN,NaN,NaN,21.100000,NaN,19.660000,NaN,26.930000,14.890000,NaN
2006-01-05,2.660000,NaN,NaN,29.680000,38.070000,38.220001,36.320000,37.910000,41.869999,NaN,...,NaN,NaN,NaN,21.290001,NaN,19.469999,NaN,28.549999,14.730000,NaN
2006-01-06,2.720000,NaN,NaN,31.150000,39.000000,38.369999,36.630001,38.810001,44.290001,NaN,...,NaN,NaN,NaN,20.959999,NaN,19.900000,NaN,29.120001,14.640000,NaN
2006-01-09,2.720000,NaN,NaN,31.059999,38.380001,38.910000,36.540001,38.799999,43.639999,NaN,...,NaN,NaN,NaN,21.590000,NaN,21.139999,NaN,29.090000,14.840000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-09-24,108.220001,24.930000,21.920000,214.419998,467.670013,112.680000,130.800003,42.320000,220.960007,9.81,...,195.520004,11.51,140.380005,202.009995,208.690002,37.529999,21.530001,99.010002,17.790001,464.980011
2020-09-25,112.279999,25.240000,21.520000,214.630005,479.779999,113.480003,134.539993,42.000000,227.800003,9.93,...,197.250000,11.56,141.649994,205.580002,215.779999,38.470001,21.570000,100.349998,17.980000,496.500000
2020-09-28,114.959999,25.700001,22.280001,222.779999,488.510010,117.059998,137.309998,43.259998,232.149994,10.21,...,200.320007,11.90,142.990005,204.720001,220.339996,38.619999,21.990000,103.800003,18.580000,487.660004


In [20]:
def extract_dates(df, d):
    d = max(df.index) if d>max(df.index) else d
    try:
        p = df.loc[d]
    except:
        d2 = (d + pd.DateOffset(days=1)).date()
        p = extract_dates(df, d2)
    return p

In [21]:
def extract_prices(df):
    prices = pd.DataFrame()
    for d in release_date:
        p = extract_dates(price_df, d)
        prices = pd.concat([prices, p], axis=1)
    return prices

In [22]:
prices = extract_prices(price_df)
prices.index.names = ['issue_code']
converter = {}
for i in range(len(term)):
    converter[prices.columns[i]] = 'price_{}'.format(term[i].strftime('%y%qQ'))
prices.rename(columns=converter, inplace=True)

In [23]:
df = df.merge(prices, on='issue_code', suffixes=('',''))
df.dropna(how='all', axis=1, inplace=True)
df

,revenue_061Q,revenue_062Q,revenue_063Q,revenue_064Q,revenue_071Q,revenue_072Q,revenue_073Q,revenue_074Q,revenue_081Q,revenue_082Q,...,price_182Q,price_183Q,price_184Q,price_191Q,price_192Q,price_193Q,price_194Q,price_201Q,price_202Q,price_203Q
issue_code,,,,,,,,,,,,,,,,,,,,,
AAPL,4359.000000,4370.000000,4837.000000,7115.000000,5264.000000,5410.000000,6789.000000,9608.000000,7512.000000,7464.000000,...,56.259998,44.650002,43.290001,43.770000,52.189999,66.040001,74.699997,80.459999,129.039993,115.809998
ACIW,89.833000,84.764000,NaN,93.289001,89.947998,98.109001,84.872002,101.281998,90.663002,109.219002,...,28.370001,28.879999,31.870001,31.459999,29.780001,36.730000,29.240000,27.730000,29.379999,26.129999
ACLS,97.920998,117.639999,122.817001,123.338997,97.526001,110.072998,107.553001,89.648003,84.893997,76.889000,...,20.200001,19.920000,21.020000,14.830000,15.310000,21.209999,23.950001,26.190001,23.629999,22.000000
ACN,4491.111816,4805.327148,4388.910156,5166.358887,5169.353027,5543.684082,5573.350098,6101.957031,6057.623047,6593.201172,...,168.630005,164.520004,161.380005,178.070007,198.169998,199.589996,189.550003,203.380005,239.929993,225.990005
ADBE,655.478027,635.455994,602.190979,682.174988,649.406982,745.577026,851.685974,911.211121,890.445007,886.885986,...,266.429993,250.889999,262.500000,270.899994,284.510010,302.750000,360.279999,389.679993,513.390015,490.429993
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WDAY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,153.809998,164.000000,197.929993,204.119995,177.279999,171.929993,172.889999,177.960007,239.710007,215.130005
WDC,1129.000000,1085.000000,1264.000000,1428.000000,1410.000000,1366.000000,1766.000000,2204.000000,2111.000000,1993.000000,...,63.279999,45.389999,50.299999,37.220001,57.270000,48.500000,60.040001,43.080002,38.419998,36.549999
WU,1043.000000,1113.599976,1140.400024,1173.199951,1131.000000,1202.900024,1257.199951,1309.099976,1265.900024,1347.099976,...,18.799999,18.730000,17.870001,19.400000,22.120001,26.910000,23.490000,20.709999,23.590000,21.430000


In [24]:
df['price_202Q']

issue_code
AAPL    129.039993
ACIW     29.379999
ACLS     23.629999
ACN     239.929993
ADBE    513.390015
           ...    
WDAY    239.710007
WDC      38.419998
WU       23.590000
XLNX    104.160004
XRX      18.860001
Name: price_202Q, Length: 261, dtype: float64

In [25]:
for y in range(2005, 2020):
    for q in range(1, 5):
        try:
            df['PER_{}{}Q'.format(str(y+1)[-2:], q)] = round( df['price_{}{}Q'.format(str(y+1)[-2:], q)] / df['EPS_{}{}Q_ttm'.format(str(y+1)[-2:], q)], 2)
            df['PBR_{}{}Q'.format(str(y+1)[-2:], q)] = round( df['price_{}{}Q'.format(str(y+1)[-2:], q)] / df['BPS_{}{}Q'.format(str(y+1)[-2:], q)], 2)
        except:
            pass

In [26]:
df['PBR_202Q']

issue_code
AAPL     30.59
ACIW      3.13
ACLS      1.78
ACN       9.27
ADBE     22.65
         ...  
WDAY     21.65
WDC       1.21
WU     -132.08
XLNX     10.97
XRX       0.71
Name: PBR_202Q, Length: 261, dtype: float64

In [27]:
df[(df['EPS_202Q']>df['EPS_192Q']) 
   & (df['EPS_202Q_ttm']>df['EPS_192Q_ttm']) 
   & (df['price_202Q']<df['price_192Q'])
  ]

,revenue_061Q,revenue_062Q,revenue_063Q,revenue_064Q,revenue_071Q,revenue_072Q,revenue_073Q,revenue_074Q,revenue_081Q,revenue_082Q,...,PER_193Q,PBR_193Q,PER_194Q,PBR_194Q,PER_201Q,PBR_201Q,PER_202Q,PBR_202Q,PER_203Q,PBR_203Q
issue_code,,,,,,,,,,,,,,,,,,,,,
CTG,83.642998,85.764999,79.830002,78.014999,80.015999,80.139999,80.625000,84.503998,86.682999,94.070999,...,-29.35,1.30,20.52,1.32,12.75,0.91,12.97,1.04,NaN,NaN
CVV,3.211500,3.111100,3.635500,3.397600,3.811300,3.071200,3.302800,3.392400,4.043500,4.268900,...,-4.14,0.70,-3.87,0.82,-8.32,0.66,-8.85,0.65,NaN,NaN
DAKT,71.050003,90.172997,92.153000,123.529999,106.731003,110.787003,120.922997,131.436005,118.200996,129.117004,...,150.75,1.41,525.00,1.20,-21.45,1.07,442.00,1.11,198.00,0.95
DBD,623.690979,735.263977,726.682007,835.337097,646.286011,695.184998,740.853027,806.026978,691.908020,768.676025,...,-1.77,-1.28,-1.69,-1.14,-1.43,-0.61,-2.34,-0.91,NaN,NaN
DMRC,NaN,NaN,NaN,NaN,NaN,NaN,3.307000,NaN,5.085000,5.115000,...,-11.52,7.10,-7.47,5.25,-6.43,5.31,-5.85,5.43,NaN,NaN
JCOM,42.018002,44.265999,45.890999,48.903999,54.140999,53.980000,55.745998,56.830002,58.647999,60.676998,...,32.25,4.20,20.37,3.25,22.04,3.01,18.71,2.60,NaN,NaN
MAGS,13.472000,14.874000,16.160000,22.406000,15.053000,14.949000,21.884001,23.155001,26.290001,25.021000,...,141.67,1.15,52.86,0.98,27.73,0.84,26.54,0.92,NaN,NaN
MEI,95.153000,116.709999,103.754997,108.667000,106.098000,131.503006,125.154999,133.766006,138.778000,155.253006,...,14.28,1.89,10.80,1.52,10.34,1.52,8.68,1.34,9.34,1.31
PCTI,18.566000,26.757999,20.525999,10.918000,16.617001,16.500000,17.625999,19.145000,18.299999,20.274000,...,-20.88,2.33,33.14,1.77,35.00,1.78,30.19,1.67,NaN,NaN


In [36]:
def eps_filter(start_year, end_year):
    for i in range(start_year, end_year):
        for q in range(1, 5):
            try:
                rst = df[( df['EPS_{}{}Q'.format(str(i+1)[-2:], q)] > df['EPS_{}{}Q'.format(str(i)[-2:], q)] ) 
                   & ( df['EPS_{}{}Q_ttm'.format(str(i+1)[-2:], q)] > df['EPS_{}{}Q_ttm'.format(str(i)[-2:], q)] ) 
                   & ( df['price_{}{}Q'.format(str(i+1)[-2:], q)] < df['price_{}{}Q'.format(str(i)[-2:], q)] )
                  ].index.tolist()
                print('{}-{}Q ({}) {}'.format(i+1, q, len(rst), rst))
            except:
                print('{}-{}Q (0)'.format(i+1, q))

In [30]:
eps_filter(2010, 2020)

2011-1Q (15) ['ACLS', 'AKAM', 'CMTL', 'COHU', 'CSGS', 'CTS', 'G', 'HOLI', 'HPQ', 'ITRI', 'MANT', 'MRVL', 'MSFT', 'RIOT', 'TAOP']
2011-2Q (18) ['ADBE', 'ADTN', 'AKAM', 'ASYS', 'AVID', 'AVNW', 'CAMT', 'CSIQ', 'EBIX', 'FFIV', 'HPQ', 'ITRI', 'ITRN', 'LDOS', 'MVIS', 'PRFT', 'UTSI', 'XRX']
2011-3Q (45) ['ACLS', 'ADI', 'ADSK', 'AEHR', 'AKAM', 'AMAT', 'APH', 'AVID', 'AVNW', 'AXTI', 'BAH', 'BDC', 'CAMT', 'CIEN', 'DAKT', 'DLB', 'DMRC', 'ENV', 'EVOL', 'FFIV', 'FIS', 'FN', 'FORM', 'GLW', 'HOLI', 'HPQ', 'III', 'INSG', 'ITRN', 'LLNW', 'MANT', 'MGIC', 'MNDO', 'MOSY', 'MVIS', 'NSYS', 'PEGA', 'PLT', 'PRFT', 'PTC', 'RBBN', 'SWKS', 'UTSI', 'WU', 'XRX']
2011-4Q (41) ['ADSK', 'AKAM', 'AVID', 'BAH', 'CACI', 'CIEN', 'CLRO', 'CSGS', 'CTSH', 'DGLY', 'EBIX', 'EVOL', 'FIS', 'GNSS', 'HLIT', 'HOLI', 'IMMR', 'IPGP', 'ITRN', 'LSCC', 'LUNA', 'MEI', 'MGIC', 'MOSY', 'MVIS', 'NLOK', 'NOVT', 'NSYS', 'NVDA', 'OLED', 'ORCL', 'PAYX', 'PCTI', 'PEGA', 'PFSW', 'PRFT', 'RIOT', 'STCN', 'UTSI', 'WU', 'XRX']
2012-1Q (47) ['ACN', '

In [31]:
def per_filter(start_year, end_year):
    for i in range(start_year, end_year):
        for q in range(1, 5):
            try:
                rst = df[( df['PER_{}{}Q'.format(str(i+1)[-2:], q)] > df['PER_{}{}Q'.format(str(i)[-2:], q)] ) 
                         
                  ].index.tolist()
                print('{}-{}Q ({}) {}'.format(i+1, q, len(rst), rst))
            except:
                print('{}-{}Q (0)'.format(i+1, q))

In [32]:
per_filter(2010, 2020)

2011-1Q (102) ['ACIW', 'ACN', 'ADP', 'AEHR', 'AEIS', 'ALOT', 'AMAT', 'AMD', 'AMKR', 'ANSS', 'ASYS', 'AVGO', 'AVNW', 'BELFA', 'BLKB', 'BMI', 'BR', 'BRKS', 'BSQR', 'CACI', 'CALX', 'CAMT', 'CCMP', 'CDNS', 'CLFD', 'CLRO', 'COHU', 'CRM', 'CSIQ', 'CSPI', 'CTG', 'CTSH', 'CTXS', 'CYRN', 'DAKT', 'DGLY', 'DMRC', 'EGAN', 'EGHT', 'ENTG', 'EPAY', 'EVOL', 'EXTR', 'FARO', 'FEIM', 'FICO', 'FISV', 'FLIR', 'FORM', 'FSLR', 'GPN', 'HCKT', 'IBM', 'IDCC', 'III', 'INOD', 'INTU', 'ISNS', 'IVAC', 'JKHY', 'KBR', 'LPSN', 'LPTH', 'LTRX', 'MA', 'MAGS', 'MANH', 'MEI', 'MGIC', 'MKSI', 'MMS', 'MNDO', 'MVIS', 'NCR', 'NLOK', 'NTCT', 'NTWK', 'NUAN', 'NVDA', 'NVEC', 'OCC', 'ONTO', 'ORCL', 'OSIS', 'PAR', 'PAYX', 'PLAB', 'PRCP', 'PRGX', 'PTC', 'QCOM', 'RIOT', 'ROG', 'SNPS', 'SNX', 'STX', 'SWKS', 'TDC', 'UIS', 'VIAV', 'WDC', 'WU']
2011-2Q (95) ['ACLS', 'ACN', 'ADP', 'ADS', 'AEHR', 'AEY', 'AMD', 'AMKR', 'ANSS', 'ASUR', 'AVNW', 'BLKB', 'BMI', 'BR', 'BSQR', 'CALX', 'CAMT', 'CCMP', 'CIEN', 'CLFD', 'CMTL', 'CRM', 'CSPI', 'CTG', 

In [33]:
def pbr_filter(start_year, end_year):
    for i in range(start_year, end_year):
        for q in range(1, 5):
            try:
                rst = df[( 0 < df['PBR_{}{}Q'.format(str(i+1)[-2:], q)] ) 
                         & ( df['PBR_{}{}Q'.format(str(i+1)[-2:], q)] < 1 )
                  ].index.tolist()
                print('{}-{}Q ({}) {}'.format(i+1, q, len(rst), rst))
            except:
                print('{}-{}Q (0)'.format(i+1, q))

In [34]:
pbr_filter(2010, 2020)

2011-1Q (19) ['ACLS', 'AEY', 'AVNW', 'BHE', 'CNXN', 'CSPI', 'DXC', 'HPQ', 'III', 'KTCC', 'MOSY', 'OCC', 'PAR', 'QUIK', 'SPWR', 'SRT', 'STCN', 'TAOP', 'XRX']
2011-2Q (33) ['ACLS', 'AEY', 'AOSL', 'ASYS', 'AVID', 'AVNW', 'BHE', 'CNXN', 'COHU', 'CSPI', 'DXC', 'HPQ', 'III', 'INSG', 'ISNS', 'KTCC', 'LLNW', 'MOSY', 'MU', 'NPTN', 'NTWK', 'OCC', 'PAR', 'PLAB', 'PRCP', 'QUIK', 'SNX', 'SPWR', 'SRT', 'STCN', 'TAOP', 'UTSI', 'XRX']
2011-3Q (40) ['ACLS', 'AEHR', 'AEY', 'AOSL', 'ASYS', 'AVID', 'AVNW', 'AXTI', 'BHE', 'CAMT', 'CNXN', 'COHU', 'CSPI', 'DXC', 'EMKR', 'FEIM', 'FORM', 'GLW', 'HPQ', 'III', 'INSG', 'ISNS', 'KTCC', 'LEDS', 'MOSY', 'MU', 'NPTN', 'NTWK', 'OCC', 'PAR', 'PLAB', 'PRCP', 'QUIK', 'SNX', 'SPWR', 'SRT', 'STCN', 'TAOP', 'UTSI', 'XRX']
2011-4Q (36) ['ACLS', 'AEHR', 'AEY', 'AOSL', 'ASYS', 'AVNW', 'BHE', 'CNXN', 'COHU', 'CSIQ', 'CSPI', 'DQ', 'DXC', 'FEIM', 'FORM', 'FSLR', 'GLW', 'HPQ', 'INSG', 'ISNS', 'IVAC', 'LEDS', 'MOSY', 'MU', 'NPTN', 'NSYS', 'NTWK', 'OCC', 'PLAB', 'PRCP', 'QUIK', 'SPW

In [35]:
df[ (0<df['PBR_202Q']) & (df['PBR_202Q']<1) ]

,revenue_061Q,revenue_062Q,revenue_063Q,revenue_064Q,revenue_071Q,revenue_072Q,revenue_073Q,revenue_074Q,revenue_081Q,revenue_082Q,...,PER_193Q,PBR_193Q,PER_194Q,PBR_194Q,PER_201Q,PBR_201Q,PER_202Q,PBR_202Q,PER_203Q,PBR_203Q
issue_code,,,,,,,,,,,,,,,,,,,,,
ALOT,16.004499,15.641500,16.266600,16.042900,17.568199,16.406900,18.694500,19.139099,18.130899,18.687901,...,17.23,1.48,15.27,1.09,27.46,0.65,97.86,0.68,-133.67,0.79
AOSL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,582.00,0.65,270.25,0.61,-64.31,0.60,-50.19,0.79,NaN,NaN
ASUR,4.351000,2.538000,NaN,9.096000,1.045000,20.978001,NaN,1.875000,2.734000,2.707000,...,-13.69,1.34,4.27,0.94,3.09,0.72,3.24,0.80,NaN,NaN
ASYS,10.892000,10.351000,11.287000,9.451000,10.539000,12.874000,13.120000,11.741000,17.591000,24.146999,...,-16.28,0.96,-14.59,0.80,-6.79,0.89,-6.17,0.97,NaN,NaN
BHE,651.244019,749.171021,769.549011,737.339783,752.481995,756.294983,672.594971,734.547119,684.309021,682.416016,...,25.17,1.22,45.93,1.00,59.75,0.79,653.67,0.73,NaN,NaN
BOSC,5.102000,4.457000,5.298000,6.060000,5.383000,5.689000,5.510000,7.192000,12.151000,13.763000,...,-68.80,NaN,-7.87,0.61,-8.04,NaN,-4.20,0.97,NaN,NaN
CMTL,95.740997,88.997002,100.206001,97.070000,111.383003,119.417000,117.814003,115.055000,152.029999,138.067993,...,35.75,1.67,24.57,1.28,18.90,0.82,33.20,0.75,50.00,0.63
CVV,3.211500,3.111100,3.635500,3.397600,3.811300,3.071200,3.302800,3.392400,4.043500,4.268900,...,-4.14,0.70,-3.87,0.82,-8.32,0.66,-8.85,0.65,NaN,NaN
ELSE,1.350000,1.420000,1.540000,1.440000,1.560000,1.600000,2.040000,1.890000,1.840000,1.710000,...,88.75,0.94,61.33,0.97,82.00,0.87,181.00,0.96,NaN,NaN


In [45]:
def pbr_filter(start_year, end_year):
    result = {}
    for i in range(start_year, end_year):
        for q in range(1, 5):
            try:
                rst = df[( 0.3 < df['PBR_{}{}Q'.format(str(i+1)[-2:], q)] ) 
                         & ( df['PBR_{}{}Q'.format(str(i+1)[-2:], q)] < 1 )
                  ].index.tolist()
                result['{}{}Q'.format(str(i+1)[-2:], q)] = rst
            except:
                pass
    return result

In [47]:
df_pbr = pbr_filter(2010, 2020)

In [50]:
df_pbr['202Q']

['ALOT',
 'AOSL',
 'ASUR',
 'ASYS',
 'BHE',
 'BOSC',
 'CMTL',
 'CVV',
 'ELSE',
 'HOLI',
 'HPE',
 'JCS',
 'JKS',
 'KTCC',
 'MAGS',
 'MTSC',
 'NSYS',
 'NTCT',
 'NTWK',
 'PLAB',
 'SRT',
 'STCN',
 'XRX']